# SSM Fraud Detection PoC — 
## IBM Credit Card Transactions · 24M Transactions · The Definitive Test

**Version 9.0** — The dataset that closes all gaps.

### Why IBM Credit Card Transactions?
| Property | Sparkov (v8.0) | IEEE-CIS (v7.5) | **IBM CC ()** |
|----------|---------------|-----------------|-------------------|
| **Transactions** | 1.85M | 590K | **24.4M** |
| **Tx per customer** | ~1,300 | ~1-10 | **~12,000** |
| **Difficulty** | Too easy (0.97+ AUPRC) | Hard but short seqs | **Hard + Long seqs** |
| **Best published F1** | 0.97+ | ~0.45 AUPRC | **0.86 (TabFormer)** |
| **Features** | Natural | Mixed/obfuscated | **Natural** |
| **Fraud rate** | 0.58% | 3.57% | **0.122%** |

### Claims Under Test (The Full Narrative)
1. **Complementarity**: SSM catches fraud that XGBoost+velocity misses
2. **Infrastructure Savings**: SSM replaces Feature Store (no velocity engineering needed)
3. **Latency**: SSM streaming inference < 10ms per transaction
4. **ATO Detection**: SSM detects behavioral shifts in transaction sequences
5. **False Positive Reduction**: Ensemble reduces FP rate vs either model alone

### Dataset Source
IBM Synthetic Credit Card Transactions (Apache 2.0)
- Paper: TabFormer (ICASSP 2021) — best F1 = 0.86
- Kaggle: ealtman2019/credit-card-transactions (152 upvotes)

## Step 0 — Environment Setup

**Before running:** Fill in your Kaggle credentials in the Configuration cell below.
The dataset will be downloaded automatically from Kaggle.

In [ ]:
# === Google Colab GPU Check ===
import torch, sys, platform
print(f"Python {sys.version}")
print(f"PyTorch {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    device = torch.device('cuda')
else:
    print("No GPU — using CPU (will be slow)")
    device = torch.device('cpu')

In [ ]:
# === Install Dependencies ===
!pip install -q xgboost scikit-learn matplotlib seaborn tqdm kaggle matplotlib-venn
import warnings; warnings.filterwarnings('ignore')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CONFIGURATION — Fill in your credentials below         ║
# ╚══════════════════════════════════════════════════════════╝

# --- Kaggle API Credentials ---
# Get your key from: https://www.kaggle.com/settings → API → Create New Token
KAGGLE_USERNAME = ""  # ← Put your Kaggle username here
KAGGLE_KEY = ""       # ← Put your Kaggle API key here

# --- Validate ---
import os
assert KAGGLE_USERNAME, "Please fill in your KAGGLE_USERNAME above!"
assert KAGGLE_KEY, "Please fill in your KAGGLE_KEY above!"

os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    import json as _j
    _j.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

print(f"✅ Kaggle API configured for user: {KAGGLE_USERNAME}")

## Step 1 — Load IBM Credit Card Transactions Dataset

The IBM dataset contains **24.4M transactions** from 2,000 synthetic consumers over decades.
Each consumer has multiple credit cards (6,139 total card accounts).

Key properties:
- **Fraud rate**: 0.122% (29,757 frauds) — extremely realistic imbalance
- **Median tx per user**: 10,860 — perfect for sequential modeling
- **Features**: Amount, MCC, Use Chip, Merchant, City, State, Zip, Time, Errors
- **License**: Apache 2.0

In [ ]:
# === Load Data — Auto-download from Kaggle ===
import pandas as pd
import numpy as np
import os, subprocess

DATA_DIR = '/content/ibm_cc_data'
DATA_PATH = os.path.join(DATA_DIR, 'credit_card_transactions-ibm_v2.csv')

if not os.path.exists(DATA_PATH):
    print('Downloading IBM Credit Card Transactions from Kaggle...')
    os.makedirs(DATA_DIR, exist_ok=True)
    subprocess.run(['kaggle', 'datasets', 'download', '-d', 'ealtman2019/credit-card-transactions',
                    '-p', DATA_DIR, '--unzip'], check=True)
    print('Downloaded!')
else:
    print('Dataset already downloaded.')

print('Loading dataset (24M+ rows — this takes ~60 seconds)...')
df_raw = pd.read_csv(DATA_PATH)
print(f"Loaded: {len(df_raw):,} transactions")
print(f"Users: {df_raw['User'].nunique()}, Cards: {df_raw.groupby(['User','Card']).ngroups}")
print(f"Fraud: {(df_raw['Is Fraud?'] == 'Yes').sum():,} ({(df_raw['Is Fraud?'] == 'Yes').mean()*100:.3f}%)")

## Step 2 — Smart Subsampling

With 24.4M transactions, training deep learning models on the full dataset would take days.
We subsample **200 users** (~2.4M transactions) which gives us:
- Enough data for robust training and evaluation
- Long sequences per user (median ~10K transactions)
- Sufficient fraud cases for statistical significance
- Tractable training time on a single GPU

In [ ]:
# === Smart Subsampling ===
np.random.seed(42)

# Ensure we include users WITH fraud
fraud_users = df_raw[df_raw['Is Fraud?'] == 'Yes']['User'].unique()
non_fraud_users = np.setdiff1d(df_raw['User'].unique(), fraud_users)

N_USERS = 200  # Configurable — increase for more data
n_fraud_users = min(len(fraud_users), N_USERS // 2)
n_other_users = N_USERS - n_fraud_users

selected_fraud = np.random.choice(fraud_users, n_fraud_users, replace=False)
selected_other = np.random.choice(non_fraud_users, n_other_users, replace=False)
selected_users = np.concatenate([selected_fraud, selected_other])

df = df_raw[df_raw['User'].isin(selected_users)].copy()
print(f"Subsampled: {len(df):,} transactions from {N_USERS} users")
print(f"Fraud: {(df['Is Fraud?'] == 'Yes').sum():,} ({(df['Is Fraud?'] == 'Yes').mean()*100:.3f}%)")
print(f"Tx per user: min={df.groupby('User').size().min()}, median={df.groupby('User').size().median():.0f}, max={df.groupby('User').size().max()}")

del df_raw  # Free memory
import gc; gc.collect()

## Step 3 — Feature Engineering

**A. Transaction-level features** (used by all models):
- Time: hour (cyclical), day of week (cyclical), month (cyclical)
- Amount: log-transformed, z-scored
- Use Chip: one-hot (Swipe / Online / Chip)
- MCC: top-20 categories one-hot + "other"
- Has Error: binary flag
- Error type: one-hot encoded

**B. Velocity features** (XGBoost only — to simulate Feature Store):
- Rolling count of transactions in 1h, 24h, 7d windows
- Rolling mean/max amount in 24h, 7d windows
- Unique merchants in 24h, 7d
- Unique MCCs in 24h
- Time since last transaction (log seconds)

The velocity features represent what a bank's **Feature Store** would compute.
SSM models must learn equivalent patterns from raw sequences alone.

In [ ]:
# === Feature Engineering ===
from sklearn.preprocessing import LabelEncoder

def parse_and_engineer(df):
    """Create all features for IBM CC dataset."""
    df = df.copy()

    # --- Parse columns ---
    df['Amount_clean'] = pd.to_numeric(
        df['Amount'].str.replace('$', '', regex=False), errors='coerce'
    ).fillna(0.0)
    df['Is_Fraud'] = (df['Is Fraud?'] == 'Yes').astype(int)

    # Build datetime
    df['datetime'] = pd.to_datetime(
        df['Year'].astype(str) + '-' + df['Month'].astype(str).str.zfill(2) + '-' +
        df['Day'].astype(str).str.zfill(2) + ' ' + df['Time'],
        errors='coerce'
    )
    df = df.dropna(subset=['datetime'])  # Drop rows with unparseable dates
    df = df.sort_values(['User', 'Card', 'datetime']).reset_index(drop=True)

    # --- Time features (cyclical) ---
    df['hour'] = df['datetime'].dt.hour
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['dow'] = df['datetime'].dt.dayofweek
    df['dow_sin'] = np.sin(2 * np.pi * df['dow'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['dow'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)

    # --- Amount features ---
    df['log_amt'] = np.log1p(df['Amount_clean'])
    amt_mean = df['log_amt'].mean()
    amt_std = df['log_amt'].std()
    df['amt_zscore'] = (df['log_amt'] - amt_mean) / (amt_std + 1e-8)

    # --- Use Chip one-hot (handle missing values) ---
    df['Use Chip'] = df['Use Chip'].fillna('Unknown')
    chip_dummies = pd.get_dummies(df['Use Chip'], prefix='chip').astype(np.float32)
    df = pd.concat([df, chip_dummies], axis=1)

    # --- MCC: top-20 + other (handle missing values) ---
    df['MCC'] = df['MCC'].fillna(0).astype(int)
    top_mcc = df['MCC'].value_counts().head(20).index
    df['MCC_cat'] = df['MCC'].where(df['MCC'].isin(top_mcc), other='other')
    mcc_dummies = pd.get_dummies(df['MCC_cat'], prefix='mcc').astype(np.float32)
    df = pd.concat([df, mcc_dummies], axis=1)

    # --- Error features ---
    # Errors? column: empty string = no error, non-empty = has error
    df['has_error'] = (df['Errors?'].fillna('').str.strip().str.len() > 0).astype(int)

    # --- Final NaN safety net ---
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    nan_counts = df[numeric_cols].isna().sum()
    if nan_counts.sum() > 0:
        print(f"WARNING: Found NaN in {nan_counts[nan_counts > 0].to_dict()}")
        print("Filling NaN with 0...")
        df[numeric_cols] = df[numeric_cols].fillna(0)

    print(f"Engineered features on {len(df):,} transactions")
    print(f"NaN check: {df.select_dtypes(include=[np.number]).isna().sum().sum()} remaining NaN values")
    return df, amt_mean, amt_std

df, amt_mean, amt_std = parse_and_engineer(df)
print(f"Fraud rate: {df['Is_Fraud'].mean()*100:.3f}%")
print(f"Columns: {len(df.columns)}")

In [ ]:
# === Velocity Features (Vectorized — FAST) ===
from tqdm import tqdm

def add_velocity_features_fast(df):
    """Compute velocity features using vectorized rolling windows.
    This simulates what a bank's Feature Store would compute in real-time."""
    df = df.copy()
    df = df.sort_values(['User', 'Card', 'datetime']).reset_index(drop=True)

    # Initialize
    vel_cols = ['vel_tx_1h', 'vel_tx_24h', 'vel_tx_7d',
                'vel_avg_amt_24h', 'vel_max_amt_24h',
                'vel_avg_amt_7d', 'vel_max_amt_7d',
                'vel_unique_mcc_24h', 'vel_unique_mcc_7d',
                'vel_time_since_last']
    for c in vel_cols:
        df[c] = 0.0

    # Process per user-card (vectorized with pandas rolling)
    for (user, card), group in tqdm(df.groupby(['User', 'Card']), desc='Computing velocity features'):
        idx = group.index
        times = group['datetime']
        amts = group['Amount_clean']
        mccs = group['MCC']

        # Time since last transaction (log seconds)
        time_diffs = times.diff().dt.total_seconds().fillna(0)
        df.loc[idx, 'vel_time_since_last'] = np.log1p(time_diffs.values)

        # For each transaction, count/aggregate in windows
        # Use a loop but vectorized inner operations
        times_arr = times.values.astype('datetime64[s]').astype(np.int64)
        amts_arr = amts.values
        mccs_arr = mccs.values

        for i in range(1, len(idx)):
            row_idx = idx[i]
            t = times_arr[i]

            # Find window boundaries
            mask_1h = (t - times_arr[:i]) <= 3600
            mask_24h = (t - times_arr[:i]) <= 86400
            mask_7d = (t - times_arr[:i]) <= 604800

            df.at[row_idx, 'vel_tx_1h'] = mask_1h.sum()
            df.at[row_idx, 'vel_tx_24h'] = mask_24h.sum()
            df.at[row_idx, 'vel_tx_7d'] = mask_7d.sum()

            if mask_24h.any():
                df.at[row_idx, 'vel_avg_amt_24h'] = amts_arr[:i][mask_24h].mean()
                df.at[row_idx, 'vel_max_amt_24h'] = amts_arr[:i][mask_24h].max()
                df.at[row_idx, 'vel_unique_mcc_24h'] = len(set(mccs_arr[:i][mask_24h]))

            if mask_7d.any():
                df.at[row_idx, 'vel_avg_amt_7d'] = amts_arr[:i][mask_7d].mean()
                df.at[row_idx, 'vel_max_amt_7d'] = amts_arr[:i][mask_7d].max()
                df.at[row_idx, 'vel_unique_mcc_7d'] = len(set(mccs_arr[:i][mask_7d]))

    return df, vel_cols

print("Computing velocity features (this takes 10-30 minutes on 2.4M rows)...")
print("TIP: These features simulate what a bank's Feature Store computes in production.")
print("     SSM models must learn equivalent patterns WITHOUT these features.")
df, velocity_cols = add_velocity_features_fast(df)
print(f"\nDone! Added {len(velocity_cols)} velocity features.")

## Step 4 — Define Feature Sets

Two feature sets enable the critical comparison:
1. **Sequence features** — SSM/RNN/Transformer get ONLY these (raw transaction data)
2. **XGBoost features** — sequence features + velocity features (simulates Feature Store)

This is the **fair test**: can SSM learn from raw sequences what XGBoost needs
explicit Feature Store engineering to capture?

In [ ]:
# === Define Feature Columns ===
# Chip columns
chip_cols = [c for c in df.columns if c.startswith('chip_')]
# MCC columns
mcc_cols = [c for c in df.columns if c.startswith('mcc_')]

# Sequence features (what SSM models see — NO velocity)
seq_feature_cols = (
    ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
     'log_amt', 'amt_zscore', 'has_error'] +
    chip_cols + mcc_cols
)

# XGBoost features (sequence + velocity = simulates Feature Store)
xgb_feature_cols = seq_feature_cols + velocity_cols

print(f"Sequence features (SSM input): {len(seq_feature_cols)}")
print(f"Velocity features (XGBoost extra): {len(velocity_cols)}")
print(f"XGBoost total features: {len(xgb_feature_cols)}")
print(f"\nSequence features: {seq_feature_cols}")
print(f"\nVelocity features: {velocity_cols}")

## Step 5 — Chronological Train / Val / Test Split

We split chronologically to prevent data leakage:
- **Train**: First 70% of each user's transactions
- **Val**: Next 15%
- **Test**: Last 15%

This mirrors real-world deployment where the model only sees past data.

In [ ]:
# === Chronological Split ===
def chronological_split(df, train_frac=0.70, val_frac=0.15):
    """Split each user's transactions chronologically."""
    train_idx, val_idx, test_idx = [], [], []

    for (user, card), group in df.groupby(['User', 'Card']):
        n = len(group)
        if n < 10:  # Skip very short sequences
            continue
        train_end = int(n * train_frac)
        val_end = int(n * (train_frac + val_frac))

        train_idx.extend(group.index[:train_end])
        val_idx.extend(group.index[train_end:val_end])
        test_idx.extend(group.index[val_end:])

    return train_idx, val_idx, test_idx

train_idx, val_idx, test_idx = chronological_split(df)

df_train = df.loc[train_idx].copy()
df_val = df.loc[val_idx].copy()
df_test = df.loc[test_idx].copy()

print(f"Train: {len(df_train):,} tx ({df_train['Is_Fraud'].sum():,} fraud, {df_train['Is_Fraud'].mean()*100:.3f}%)")
print(f"Val:   {len(df_val):,} tx ({df_val['Is_Fraud'].sum():,} fraud, {df_val['Is_Fraud'].mean()*100:.3f}%)")
print(f"Test:  {len(df_test):,} tx ({df_test['Is_Fraud'].sum():,} fraud, {df_test['Is_Fraud'].mean()*100:.3f}%)")
print(f"\nTrain period: {df_train['datetime'].min()} → {df_train['datetime'].max()}")
print(f"Test  period: {df_test['datetime'].min()} → {df_test['datetime'].max()}")

## Step 6 — Build Transaction Sequences

For each transaction, we build a sequence of the **last SEQ_LEN transactions**
from the same user-card pair. This is what SSM models process.

With IBM's long histories (~12K tx per user), we can use SEQ_LEN=50
and have **real data** filling the entire window (unlike IEEE-CIS where most was padding).

In [ ]:
# === Sequence Building ===
SEQ_LEN = 50  # Configurable: 50, 100, 200

def build_sequences(df, feature_cols, seq_len=50):
    """Build sequences of last seq_len transactions per user-card."""
    sequences = []
    labels = []
    flat_indices = []

    for (user, card), group in tqdm(df.groupby(['User', 'Card']), desc='Building sequences'):
        group = group.sort_values('datetime')
        features = group[feature_cols].values.astype(np.float32)
        features = np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)  # NaN safety
        fraud_labels = group['Is_Fraud'].values
        indices = group.index.values

        for i in range(len(group)):
            start = max(0, i - seq_len + 1)
            seq = features[start:i+1]

            # Pad if needed
            if len(seq) < seq_len:
                pad = np.zeros((seq_len - len(seq), len(feature_cols)), dtype=np.float32)
                seq = np.vstack([pad, seq])

            sequences.append(seq)
            labels.append(fraud_labels[i])
            flat_indices.append(indices[i])

    return np.array(sequences), np.array(labels), np.array(flat_indices)

print(f"Building sequences with SEQ_LEN={SEQ_LEN}...")
print("\nTrain sequences...")
X_train_seq, y_train, idx_train = build_sequences(df_train, seq_feature_cols, SEQ_LEN)
print("Val sequences...")
X_val_seq, y_val, idx_val = build_sequences(df_val, seq_feature_cols, SEQ_LEN)
print("Test sequences...")
X_test_seq, y_test, idx_test = build_sequences(df_test, seq_feature_cols, SEQ_LEN)

print(f"\nTrain: {X_train_seq.shape} ({y_train.sum():,} fraud)")
print(f"Val:   {X_val_seq.shape} ({y_val.sum():,} fraud)")
print(f"Test:  {X_test_seq.shape} ({y_test.sum():,} fraud)")
print(f"\nInput dim per timestep: {X_train_seq.shape[2]}")

## Step 7 — XGBoost Baselines

Two XGBoost models:
1. **XGBoost (basic)** — same features as SSM (no velocity)
2. **XGBoost (+velocity)** — with rolling window features (simulates Feature Store)

If SSM catches fraud that XGBoost+velocity misses, it proves SSM learns patterns
**beyond** what hand-crafted Feature Store features capture.

In [ ]:
# === XGBoost Training ===
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

# Prepare flat data
X_train_basic = np.nan_to_num(df_train[seq_feature_cols].values.astype(np.float32))
X_val_basic = np.nan_to_num(df_val[seq_feature_cols].values.astype(np.float32))
X_test_basic = np.nan_to_num(df_test[seq_feature_cols].values.astype(np.float32))

X_train_vel = np.nan_to_num(df_train[xgb_feature_cols].values.astype(np.float32))
X_val_vel = np.nan_to_num(df_val[xgb_feature_cols].values.astype(np.float32))
X_test_vel = np.nan_to_num(df_test[xgb_feature_cols].values.astype(np.float32))

y_train_flat = df_train['Is_Fraud'].values
y_val_flat = df_val['Is_Fraud'].values
y_test_flat = df_test['Is_Fraud'].values

fraud_ratio = (y_train_flat == 0).sum() / max((y_train_flat == 1).sum(), 1)
print(f"Class ratio (neg/pos): {fraud_ratio:.1f}")

# --- XGBoost Basic ---
print("\nTraining XGBoost (basic features)...")
xgb_basic = XGBClassifier(
    n_estimators=500, max_depth=8, learning_rate=0.05,
    scale_pos_weight=fraud_ratio, eval_metric='aucpr',
    tree_method='hist', random_state=42, early_stopping_rounds=30
)
xgb_basic.fit(X_train_basic, y_train_flat,
              eval_set=[(X_val_basic, y_val_flat)], verbose=False)
xgb_basic_probs_flat = xgb_basic.predict_proba(X_test_basic)[:, 1]
print(f"XGBoost (basic) — AUPRC: {average_precision_score(y_test_flat, xgb_basic_probs_flat):.4f}")

# --- XGBoost + Velocity ---
print("\nTraining XGBoost (+ velocity features)...")
xgb_vel = XGBClassifier(
    n_estimators=500, max_depth=8, learning_rate=0.05,
    scale_pos_weight=fraud_ratio, eval_metric='aucpr',
    tree_method='hist', random_state=42, early_stopping_rounds=30
)
xgb_vel.fit(X_train_vel, y_train_flat,
            eval_set=[(X_val_vel, y_val_flat)], verbose=False)
xgb_vel_probs_flat = xgb_vel.predict_proba(X_test_vel)[:, 1]
print(f"XGBoost (+velocity) — AUPRC: {average_precision_score(y_test_flat, xgb_vel_probs_flat):.4f}")

# --- Feature importance ---
fi = pd.Series(xgb_vel.feature_importances_, index=xgb_feature_cols).sort_values(ascending=False)
print(f"\nTop 15 features (XGBoost +velocity):")
print(fi.head(15).to_string())

# Map to sequence indices for comparison
xgb_basic_probs = xgb_basic_probs_flat[np.searchsorted(df_test.index.values, idx_test)]
xgb_vel_probs = xgb_vel_probs_flat[np.searchsorted(df_test.index.values, idx_test)]

## Step 8 — Deep Learning Models

Four architectures, all processing sequences of length 50:
1. **Mamba (SSM)** — Selective State Space with Conv1D + expand=2
2. **Griffin (RG-LRU + Local Attention)** — Hybrid recurrent
3. **GRU (RNN)** — Classical recurrent baseline
4. **Transformer** — Attention-based baseline

In [ ]:
# === PyTorch Setup ===
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# Convert to tensors (with NaN safety)
X_train_seq = np.nan_to_num(X_train_seq, nan=0.0, posinf=0.0, neginf=0.0)
X_val_seq = np.nan_to_num(X_val_seq, nan=0.0, posinf=0.0, neginf=0.0)
X_test_seq = np.nan_to_num(X_test_seq, nan=0.0, posinf=0.0, neginf=0.0)
print(f"NaN in train seqs: {np.isnan(X_train_seq).sum()}, val: {np.isnan(X_val_seq).sum()}, test: {np.isnan(X_test_seq).sum()}")

X_train_t = torch.tensor(X_train_seq, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val_seq, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)
X_test_t = torch.tensor(X_test_seq, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

train_ds = TensorDataset(X_train_t, y_train_t)
val_ds = TensorDataset(X_val_t, y_val_t)
test_ds = TensorDataset(X_test_t, y_test_t)

BATCH_SIZE = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE*2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE*2)

INPUT_DIM = X_train_seq.shape[2]
D_MODEL = 128
N_LAYERS = 4
DROPOUT = 0.1

print(f"Input dim: {INPUT_DIM}, d_model: {D_MODEL}, layers: {N_LAYERS}")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

In [ ]:
# === Mamba Block (Full Architecture with Conv1D + Expand) ===
class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.d_inner = d_model * expand
        self.d_state = d_state

        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, d_conv,
                                padding=d_conv-1, groups=self.d_inner)
        self.x_proj = nn.Linear(self.d_inner, d_state * 2 + 1, bias=False)
        self.dt_proj = nn.Linear(1, self.d_inner, bias=True)

        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).expand(self.d_inner, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def forward(self, x):
        B, L, _ = x.shape
        xz = self.in_proj(x)
        x_part, z = xz.chunk(2, dim=-1)

        x_conv = x_part.transpose(1, 2)
        x_conv = self.conv1d(x_conv)[:, :, :L]
        x_conv = x_conv.transpose(1, 2)
        x_conv = F.silu(x_conv)

        x_proj = self.x_proj(x_conv)
        dt = x_proj[..., :1]
        B_param = x_proj[..., 1:1+self.d_state]
        C_param = x_proj[..., 1+self.d_state:]

        dt = F.softplus(self.dt_proj(dt))
        A = -torch.exp(self.A_log)

        # Selective scan
        h = torch.zeros(B, self.d_inner, self.d_state, device=x.device)
        outputs = []
        for t in range(L):
            dA = torch.exp(dt[:, t].unsqueeze(-1) * A)
            dB = dt[:, t].unsqueeze(-1) * B_param[:, t].unsqueeze(1)
            h = h * dA + dB * x_conv[:, t].unsqueeze(-1)
            y_t = (h * C_param[:, t].unsqueeze(1)).sum(-1)
            outputs.append(y_t)

        y = torch.stack(outputs, dim=1)
        y = y + self.D * x_conv
        y = y * F.silu(z)
        return self.out_proj(y)

    def forward_single_tx(self, x_t, state):
        """Process single timestep for streaming inference."""
        if state is None:
            state = {'h': torch.zeros(1, self.d_inner, self.d_state, device=x_t.device),
                     'conv_buf': torch.zeros(1, self.d_inner, self.conv1d.kernel_size[0]-1, device=x_t.device)}

        xz = self.in_proj(x_t.unsqueeze(1))
        x_part, z = xz.chunk(2, dim=-1)

        x_conv_in = x_part.transpose(1, 2)
        conv_input = torch.cat([state['conv_buf'], x_conv_in], dim=2)
        state['conv_buf'] = conv_input[:, :, 1:]
        x_conv_out = self.conv1d(conv_input)[:, :, -1:]
        x_conv_out = x_conv_out.transpose(1, 2)
        x_conv_out = F.silu(x_conv_out)

        x_proj = self.x_proj(x_conv_out.squeeze(1))
        dt = x_proj[..., :1]
        B_param = x_proj[..., 1:1+self.d_state]
        C_param = x_proj[..., 1+self.d_state:]

        dt = F.softplus(self.dt_proj(dt.unsqueeze(1))).squeeze(1)
        A = -torch.exp(self.A_log)

        dA = torch.exp(dt.unsqueeze(-1) * A)
        dB = dt.unsqueeze(-1) * B_param.unsqueeze(1)
        state['h'] = state['h'] * dA + dB * x_conv_out.squeeze(1).unsqueeze(-1)
        y_t = (state['h'] * C_param.unsqueeze(1)).sum(-1)
        y_t = y_t + self.D * x_conv_out.squeeze(1)
        y_t = y_t * F.silu(z.squeeze(1))
        out = self.out_proj(y_t)
        return out, state


class MambaFraudDetector(nn.Module):
    def __init__(self, input_dim, d_model=128, n_layers=4, d_state=16, d_conv=4, expand=2, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.layers = nn.ModuleList([MambaBlock(d_model, d_state, d_conv, expand) for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))

    def forward(self, x, lengths=None):
        x = self.input_proj(x)
        for layer, norm in zip(self.layers, self.norms):
            x = x + self.dropout(layer(norm(x)))
        return self.head(x[:, -1]).squeeze(-1)

    def forward_single_tx(self, x_t, states):
        if states is None:
            states = [None] * len(self.layers)
        x = self.input_proj(x_t)
        new_states = []
        for i, (layer, norm) in enumerate(zip(self.layers, self.norms)):
            out, s = layer.forward_single_tx(norm(x), states[i])
            x = x + out
            new_states.append(s)
        logit = self.head(x).squeeze(-1)
        return logit, new_states

print("Mamba model defined (Conv1D + expand=2)")

In [ ]:
# === Griffin Block (RG-LRU + Local Attention) ===
class RGLRUCell(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.Wa = nn.Linear(d_model, d_model, bias=True)
        self.Wx = nn.Linear(d_model, d_model, bias=True)

    def forward(self, x):
        B, L, D = x.shape
        h = torch.zeros(B, D, device=x.device)
        outputs = []
        for t in range(L):
            a = torch.sigmoid(self.Wa(x[:, t]))
            h = a * h + (1 - a) * self.Wx(x[:, t])
            outputs.append(h)
        return torch.stack(outputs, dim=1)

    def forward_single_tx(self, x_t, h):
        if h is None:
            h = torch.zeros(1, x_t.shape[-1], device=x_t.device)
        a = torch.sigmoid(self.Wa(x_t))
        h = a * h + (1 - a) * self.Wx(x_t)
        return h, h


class LocalAttention(nn.Module):
    def __init__(self, d_model, n_heads=4, window=8):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.window = window

    def forward(self, x, mask=None):
        B, L, D = x.shape
        attn_mask = torch.ones(L, L, dtype=torch.bool, device=x.device)
        for i in range(L):
            start = max(0, i - self.window + 1)
            attn_mask[i, start:i+1] = False
        out, _ = self.attn(x, x, x, attn_mask=attn_mask)
        return out


class GriffinBlock(nn.Module):
    def __init__(self, d_model, n_heads=4, window=8):
        super().__init__()
        self.rglru = RGLRUCell(d_model)
        self.local_attn = LocalAttention(d_model, n_heads, window)
        self.gate = nn.Linear(d_model * 2, d_model)

    def forward(self, x, mask=None):
        r = self.rglru(x)
        a = self.local_attn(x, mask)
        g = torch.sigmoid(self.gate(torch.cat([r, a], dim=-1)))
        return g * r + (1 - g) * a

    def forward_single_tx(self, x_t, state):
        if state is None:
            state = {'h': None, 'window_buf': []}
        h, new_h = self.rglru.forward_single_tx(x_t, state['h'])
        state['h'] = new_h
        state['window_buf'].append(x_t)
        if len(state['window_buf']) > self.local_attn.window:
            state['window_buf'] = state['window_buf'][-self.local_attn.window:]
        buf = torch.stack(state['window_buf'], dim=1)
        a_out, _ = self.local_attn.attn(x_t.unsqueeze(1), buf, buf)
        a_out = a_out.squeeze(1)
        g = torch.sigmoid(self.gate(torch.cat([h, a_out], dim=-1)))
        out = g * h + (1 - g) * a_out
        return out, state


class GriffinFraudDetector(nn.Module):
    def __init__(self, input_dim, d_model=128, n_layers=4, n_heads=4, window=8, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.layers = nn.ModuleList([GriffinBlock(d_model, n_heads, window) for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))

    def forward(self, x, lengths=None):
        mask = None
        if lengths is not None:
            B, L = x.shape[:2]
            mask = torch.arange(L, device=x.device).unsqueeze(0) >= (L - lengths.unsqueeze(1))
            mask = ~mask
        x = self.input_proj(x)
        for layer, norm in zip(self.layers, self.norms):
            x = x + self.dropout(layer(norm(x), mask))
        return self.head(x[:, -1]).squeeze(-1)

    def forward_single_tx(self, x_t, states):
        if states is None:
            states = [None] * len(self.layers)
        x = self.input_proj(x_t)
        new_states = []
        for i, (layer, norm) in enumerate(zip(self.layers, self.norms)):
            out, s = layer.forward_single_tx(norm(x), states[i])
            x = x + out
            new_states.append(s)
        logit = self.head(x).squeeze(-1)
        return logit, new_states

print("Griffin model defined (RG-LRU + Local Attention)")

In [ ]:
# === GRU Model ===
class GRUFraudDetector(nn.Module):
    def __init__(self, input_dim, d_model=128, n_layers=4, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.gru = nn.GRU(d_model, d_model, num_layers=n_layers,
                          batch_first=True, dropout=dropout if n_layers > 1 else 0)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))

    def forward(self, x, lengths=None):
        x = self.input_proj(x)
        out, _ = self.gru(x)
        return self.head(out[:, -1]).squeeze(-1)

    def forward_single_tx(self, x_t, state):
        x = self.input_proj(x_t).unsqueeze(1)
        out, state = self.gru(x, state)
        logit = self.head(out.squeeze(1)).squeeze(-1)
        return logit, state


# === Transformer Model ===
class TransformerFraudDetector(nn.Module):
    def __init__(self, input_dim, d_model=128, n_layers=4, n_heads=4, dropout=0.1, max_len=200):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

        encoder_layer = nn.TransformerEncoderLayer(d_model, n_heads, d_model*4, dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))

    def forward(self, x, lengths=None):
        B, L, _ = x.shape
        x = self.input_proj(x) + self.pe[:, :L]
        causal_mask = nn.Transformer.generate_square_subsequent_mask(L).to(x.device)
        pad_mask = None
        if lengths is not None:
            pad_mask = torch.arange(L, device=x.device).unsqueeze(0) < (L - lengths.unsqueeze(1))
        x = self.encoder(x, mask=causal_mask, src_key_padding_mask=pad_mask)
        return self.head(x[:, -1]).squeeze(-1)

print("GRU and Transformer models defined")

## Step 9 — Training Loop

Training with:
- **BCEWithLogitsLoss** with pos_weight (class imbalance correction)
- **AdamW** optimizer with cosine warmup schedule
- **Early stopping** on validation AUPRC (patience=10)
- **Gradient clipping** at 1.0

In [ ]:
# === Training Infrastructure ===
from sklearn.metrics import average_precision_score
import time

def train_model(model, name, train_loader, val_loader, epochs=60, lr=1e-3, patience=10, use_lengths=False):
    model = model.to(device)
    pos_w = torch.tensor([(y_train == 0).sum() / max((y_train == 1).sum(), 1)], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    # Cosine warmup
    warmup_epochs = 5
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return epoch / warmup_epochs
        return 0.5 * (1 + np.cos(np.pi * (epoch - warmup_epochs) / (epochs - warmup_epochs)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    best_val_auprc = 0
    best_state = None
    patience_counter = 0
    train_start = time.time()

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            xb, yb = batch[0].to(device), batch[1].to(device)
            optimizer.zero_grad()
            if use_lengths:
                lens_b = torch.full((xb.shape[0],), SEQ_LEN, device=device)
                logits = model(xb, lens_b)
            else:
                logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        # Validation
        model.eval()
        val_probs = []
        val_labels = []
        with torch.no_grad():
            for batch in val_loader:
                xb, yb = batch[0].to(device), batch[1].to(device)
                if use_lengths:
                    lens_b = torch.full((xb.shape[0],), SEQ_LEN, device=device)
                    logits = model(xb, lens_b)
                else:
                    logits = model(xb)
                val_probs.extend(torch.sigmoid(logits).cpu().numpy())
                val_labels.extend(yb.cpu().numpy())

        val_auprc = average_precision_score(val_labels, val_probs)
        elapsed = time.time() - train_start

        if val_auprc > best_val_auprc:
            best_val_auprc = val_auprc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
            marker = ' ★'
        else:
            patience_counter += 1
            marker = ''

        if (epoch + 1) % 5 == 0 or marker:
            print(f"  [{name}] Epoch {epoch+1:3d} | Loss {total_loss/len(train_loader):.4f} | Val AUPRC {val_auprc:.4f}{marker} | {elapsed:.0f}s")

        if patience_counter >= patience:
            print(f"  [{name}] Early stop at epoch {epoch+1}")
            break

    model.load_state_dict(best_state)
    model = model.to(device)
    train_time = time.time() - train_start
    print(f"  [{name}] Best Val AUPRC: {best_val_auprc:.4f} | Training time: {train_time:.0f}s")
    return model, best_val_auprc, train_time

In [ ]:
# === Train All Models ===
print("=" * 70)
print("  TRAINING ALL MODELS")
print("=" * 70)

models = {}

# Mamba
print("\n--- Mamba (SSM) ---")
mamba = MambaFraudDetector(INPUT_DIM, D_MODEL, N_LAYERS, d_state=16, d_conv=4, expand=2, dropout=DROPOUT)
mamba, mamba_val, mamba_time = train_model(mamba, 'Mamba', train_loader, val_loader, epochs=60, use_lengths=False)
models['Mamba'] = mamba

# Griffin
print("\n--- Griffin (RG-LRU + Local Attention) ---")
griffin = GriffinFraudDetector(INPUT_DIM, D_MODEL, N_LAYERS, n_heads=4, window=8, dropout=DROPOUT)
griffin, griffin_val, griffin_time = train_model(griffin, 'Griffin', train_loader, val_loader, epochs=60, use_lengths=True)
models['Griffin'] = griffin

# GRU
print("\n--- GRU (RNN) ---")
gru = GRUFraudDetector(INPUT_DIM, D_MODEL, N_LAYERS, dropout=DROPOUT)
gru, gru_val, gru_time = train_model(gru, 'GRU', train_loader, val_loader, epochs=60, use_lengths=False)
models['GRU'] = gru

# Transformer
print("\n--- Transformer ---")
transformer = TransformerFraudDetector(INPUT_DIM, D_MODEL, N_LAYERS, n_heads=4, dropout=DROPOUT, max_len=max(SEQ_LEN, 200))
transformer, transformer_val, transformer_time = train_model(transformer, 'Transformer', train_loader, val_loader, epochs=60, use_lengths=True)
models['Transformer'] = transformer

print("\n" + "=" * 70)
print("  TRAINING COMPLETE")
print("=" * 70)

## Step 10 — Evaluation on Test Set

Evaluate all models with:
- **AUPRC** (primary metric) with Bootstrap 95% CI
- **AUROC** (for comparison with published benchmarks)
- **Precision / Recall @ 1% FPR**

In [ ]:
# === Get Test Predictions ===
def get_predictions(model, loader, use_lengths=False):
    model.eval()
    probs = []
    with torch.no_grad():
        for batch in loader:
            xb = batch[0].to(device)
            if use_lengths:
                lens_b = torch.full((xb.shape[0],), SEQ_LEN, device=device)
                logits = model(xb, lens_b)
            else:
                logits = model(xb)
            probs.extend(torch.sigmoid(logits).cpu().numpy())
    return np.array(probs)

print("Getting test predictions...")
pred_mamba = get_predictions(mamba, test_loader, use_lengths=False)
pred_griffin = get_predictions(griffin, test_loader, use_lengths=True)
pred_gru = get_predictions(gru, test_loader, use_lengths=False)
pred_transformer = get_predictions(transformer, test_loader, use_lengths=True)
print("Done!")

In [ ]:
# === Bootstrap CI + Metrics ===
from sklearn.metrics import precision_recall_curve, roc_curve

def bootstrap_auprc(y_true, y_score, n_boot=1000, ci=0.95):
    rng = np.random.RandomState(42)
    scores = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.choice(n, n, replace=True)
        if y_true[idx].sum() == 0:
            continue
        scores.append(average_precision_score(y_true[idx], y_score[idx]))
    lo = np.percentile(scores, (1-ci)/2*100)
    hi = np.percentile(scores, (1+ci)/2*100)
    return lo, hi

def precision_at_fpr(y_true, y_score, target_fpr=0.01):
    fpr_array, tpr_array, thresholds = roc_curve(y_true, y_score)
    valid_indices = np.where(fpr_array <= target_fpr)[0]
    if len(valid_indices) == 0:
        return 0, 0, 1.0
    best_idx = valid_indices[-1]
    best_threshold = thresholds[best_idx]
    preds = (y_score >= best_threshold).astype(int)
    tp = ((preds == 1) & (y_true == 1)).sum()
    fp = ((preds == 1) & (y_true == 0)).sum()
    precision = tp / (tp + fp + 1e-10)
    recall = tpr_array[best_idx]
    return precision, recall, best_threshold

# Evaluate all
all_results = {}
model_preds = {
    'Mamba (SSM)': pred_mamba,
    'Griffin (RG-LRU)': pred_griffin,
    'GRU (RNN)': pred_gru,
    'Transformer': pred_transformer,
    'XGBoost (basic)': xgb_basic_probs,
    'XGBoost (+velocity)': xgb_vel_probs,
}

print("=" * 90)
print(f"  TEST RESULTS — IBM Credit Card Transactions (N_USERS={N_USERS}, SEQ_LEN={SEQ_LEN})")
print("=" * 90)
print(f"{'Model':<25} {'AUPRC':>8} {'95% CI':>16} {'AUROC':>8} {'P@1%FPR':>9} {'R@1%FPR':>9}")
print("-" * 90)

for name, probs in model_preds.items():
    auprc = average_precision_score(y_test, probs)
    auroc = roc_auc_score(y_test, probs)
    ci_lo, ci_hi = bootstrap_auprc(y_test, probs)
    p1, r1, _ = precision_at_fpr(y_test, probs, 0.01)

    all_results[name] = {
        'auprc': auprc, 'auroc': auroc, 'ci_lo': ci_lo, 'ci_hi': ci_hi,
        'p_at_1fpr': p1, 'r_at_1fpr': r1, 'probs': probs
    }
    print(f"  {name:<23} {auprc:>8.4f} [{ci_lo:.4f}, {ci_hi:.4f}] {auroc:>8.4f} {p1:>9.4f} {r1:>9.4f}")

print("=" * 90)

## Step 11 — Ensemble Optimization

Optimize weighted ensemble (SSM + XGBoost) on validation set.
Compare against XGBoost **with velocity features** — the fair baseline.

In [ ]:
# === Ensemble Optimization ===
pred_mamba_val = get_predictions(mamba, val_loader, use_lengths=False)
pred_griffin_val = get_predictions(griffin, val_loader, use_lengths=True)

xgb_vel_probs_val = xgb_vel.predict_proba(X_val_vel)[:, 1]
xgb_vel_probs_val_seq = xgb_vel_probs_val[np.searchsorted(df_val.index.values, idx_val)]

# Optimize Mamba + XGBoost(+vel)
best_mamba_w, best_mamba_ens_auprc = 0, 0
for w in np.arange(0.05, 0.96, 0.05):
    ens = w * pred_mamba_val + (1 - w) * xgb_vel_probs_val_seq
    score = average_precision_score(y_val, ens)
    if score > best_mamba_ens_auprc:
        best_mamba_w = w
        best_mamba_ens_auprc = score

# Optimize Griffin + XGBoost(+vel)
best_griffin_w, best_griffin_ens_auprc = 0, 0
for w in np.arange(0.05, 0.96, 0.05):
    ens = w * pred_griffin_val + (1 - w) * xgb_vel_probs_val_seq
    score = average_precision_score(y_val, ens)
    if score > best_griffin_ens_auprc:
        best_griffin_w = w
        best_griffin_ens_auprc = score

print(f"Mamba+XGB(vel) optimal weight: {best_mamba_w:.2f} / {1-best_mamba_w:.2f} (val AUPRC: {best_mamba_ens_auprc:.4f})")
print(f"Griffin+XGB(vel) optimal weight: {best_griffin_w:.2f} / {1-best_griffin_w:.2f} (val AUPRC: {best_griffin_ens_auprc:.4f})")

# Test ensembles
ens_mamba_test = best_mamba_w * pred_mamba + (1 - best_mamba_w) * xgb_vel_probs
ens_griffin_test = best_griffin_w * pred_griffin + (1 - best_griffin_w) * xgb_vel_probs

for ens_name, ens_probs in [('Mamba+XGB(vel)', ens_mamba_test), ('Griffin+XGB(vel)', ens_griffin_test)]:
    auprc = average_precision_score(y_test, ens_probs)
    auroc = roc_auc_score(y_test, ens_probs)
    ci_lo, ci_hi = bootstrap_auprc(y_test, ens_probs)
    p1, r1, _ = precision_at_fpr(y_test, ens_probs, 0.01)
    all_results[ens_name] = {
        'auprc': auprc, 'auroc': auroc, 'ci_lo': ci_lo, 'ci_hi': ci_hi,
        'p_at_1fpr': p1, 'r_at_1fpr': r1, 'probs': ens_probs
    }
    print(f"  {ens_name}: AUPRC={auprc:.4f} [{ci_lo:.4f},{ci_hi:.4f}] P@1%FPR={p1:.4f} R@1%FPR={r1:.4f}")

## Step 12 — Complementarity Analysis (Claim #1)

> **Claim**: SSM catches fraud that XGBoost+velocity misses.

This is the central analysis. We compare against **XGBoost (+velocity)** — the strongest
baseline that simulates a full Feature Store — to ensure any complementarity is real.

In [ ]:
# === Part 1: Overlap Analysis at Multiple FPR Levels ===
import matplotlib.pyplot as plt

def threshold_at_fpr(y_true, y_score, target_fpr):
    fpr_array, _, thresholds = roc_curve(y_true, y_score)
    valid_indices = np.where(fpr_array <= target_fpr)[0]
    if len(valid_indices) == 0:
        return 1.0
    return thresholds[valid_indices[-1]]

total_fraud = int(y_test.sum())
print(f"Total fraud in test: {total_fraud}")

models_to_compare = {
    'Mamba': pred_mamba,
    'Griffin': pred_griffin,
    'GRU': pred_gru,
    'XGBoost (+vel)': xgb_vel_probs,
}

complementarity_data = {}

for target_fpr in [0.01, 0.02, 0.05, 0.10]:
    print(f"\n{'='*60}")
    print(f"  @ {target_fpr*100:.0f}% FPR")
    print(f"{'='*60}")

    caught = {}
    for name, probs in models_to_compare.items():
        thr = threshold_at_fpr(y_test, probs, target_fpr)
        preds = probs >= thr
        fraud_caught = set(np.where((preds == 1) & (y_test == 1))[0])
        caught[name] = fraud_caught
        print(f"  {name}: {len(fraud_caught)} caught ({len(fraud_caught)/total_fraud*100:.1f}% recall)")

    # Mamba vs XGBoost(+vel)
    m = caught['Mamba']
    x = caught['XGBoost (+vel)']
    both = m & x
    only_m = m - x
    only_x = x - m
    union = m | x

    print(f"\n  Mamba vs XGBoost(+vel):")
    print(f"    Both catch:    {len(both)}")
    print(f"    Only Mamba:    {len(only_m)} ← SSM unique catches!")
    print(f"    Only XGBoost:  {len(only_x)}")
    print(f"    Union:         {len(union)} ({len(union)/total_fraud*100:.1f}% recall)")
    print(f"    Union uplift:  +{len(only_m)} over XGBoost alone (+{len(only_m)/total_fraud*100:.1f}%)")

    # Triple union
    g = caught['Griffin']
    triple = m | g | x
    print(f"\n  Triple union (Mamba+Griffin+XGB): {len(triple)} ({len(triple)/total_fraud*100:.1f}% recall)")

    fpr_key = f'{target_fpr*100:.0f}pct_fpr'
    complementarity_data[fpr_key] = {
        'mamba_caught': len(m), 'xgb_caught': len(x), 'griffin_caught': len(g),
        'both_mamba_xgb': len(both), 'only_mamba': len(only_m), 'only_xgb': len(only_x),
        'union_mamba_xgb': len(union), 'triple_union': len(triple),
        'union_recall': len(union)/total_fraud, 'triple_recall': len(triple)/total_fraud,
    }

In [ ]:
# === Part 2: Venn Diagrams ===
try:
    from matplotlib_venn import venn2
    has_venn = True
except ImportError:
    has_venn = False

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, target_fpr in enumerate([0.01, 0.05, 0.10]):
    thr_m = threshold_at_fpr(y_test, pred_mamba, target_fpr)
    thr_x = threshold_at_fpr(y_test, xgb_vel_probs, target_fpr)
    m_set = set(np.where((pred_mamba >= thr_m) & (y_test == 1))[0])
    x_set = set(np.where((xgb_vel_probs >= thr_x) & (y_test == 1))[0])

    if has_venn:
        venn2([m_set, x_set], set_labels=('Mamba', 'XGBoost(+vel)'), ax=axes[i])
    else:
        only_m = len(m_set - x_set)
        only_x = len(x_set - m_set)
        both = len(m_set & x_set)
        axes[i].bar(['Only Mamba', 'Both', 'Only XGB'], [only_m, both, only_x],
                     color=['#2196F3', '#9C27B0', '#FF9800'])
        for j, v in enumerate([only_m, both, only_x]):
            axes[i].text(j, v+1, str(v), ha='center', fontweight='bold')

    axes[i].set_title(f'@ {target_fpr*100:.0f}% FPR (Fraud Catches)')

plt.suptitle('Mamba vs XGBoost(+velocity) — Fraud Catch Overlap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('complementarity_venn.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Part 3: Score Correlation on Fraud Cases ===
import seaborn as sns

fraud_mask = y_test == 1
fraud_scores = pd.DataFrame({
    'Mamba': pred_mamba[fraud_mask],
    'Griffin': pred_griffin[fraud_mask],
    'XGBoost(+vel)': xgb_vel_probs[fraud_mask],
})

corr = fraud_scores.corr()
print("Score correlations on FRAUD cases:")
print(corr.to_string())

corr_mx = float(corr.loc['Mamba', 'XGBoost(+vel)'])
corr_gx = float(corr.loc['Griffin', 'XGBoost(+vel)'])
corr_mg = float(corr.loc['Mamba', 'Griffin'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(xgb_vel_probs[~fraud_mask], pred_mamba[~fraud_mask], alpha=0.01, s=1, c='gray', label='Legit')
ax.scatter(xgb_vel_probs[fraud_mask], pred_mamba[fraud_mask], alpha=0.3, s=10, c='red', label='Fraud')
ax.set_xlabel('XGBoost (+velocity) score')
ax.set_ylabel('Mamba score')
ax.set_title(f'Score Agreement (r={corr_mx:.3f} on fraud)')
ax.legend()
ax.plot([0,1],[0,1], 'k--', alpha=0.3)

ax = axes[1]
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdYlBu_r', vmin=0, vmax=1, ax=ax)
ax.set_title('Score Correlation on Fraud Cases')

plt.tight_layout()
plt.savefig('score_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 13 — Infrastructure Savings Analysis (Claim #2)

> **Claim**: SSM replaces the Feature Store — no velocity engineering needed.

We compare:
- **XGBoost (basic)** — no velocity features
- **XGBoost (+velocity)** — with Feature Store simulation
- **Mamba (SSM)** — raw sequences only, no velocity

If Mamba ≥ XGBoost(+velocity), SSM effectively replaces the Feature Store.

In [ ]:
# === Infrastructure Savings Analysis ===
print("=" * 70)
print("  INFRASTRUCTURE SAVINGS — SSM vs Feature Store")
print("=" * 70)

xgb_basic_auprc = all_results['XGBoost (basic)']['auprc']
xgb_vel_auprc = all_results['XGBoost (+velocity)']['auprc']
mamba_auprc = all_results['Mamba (SSM)']['auprc']
griffin_auprc = all_results['Griffin (RG-LRU)']['auprc']

velocity_uplift = xgb_vel_auprc - xgb_basic_auprc
mamba_vs_xgb_vel = mamba_auprc - xgb_vel_auprc

print(f"\n  XGBoost (basic):     AUPRC = {xgb_basic_auprc:.4f}")
print(f"  XGBoost (+velocity): AUPRC = {xgb_vel_auprc:.4f}  (+{velocity_uplift:.4f} from Feature Store)")
print(f"  Mamba (raw seq):     AUPRC = {mamba_auprc:.4f}  ({'+' if mamba_vs_xgb_vel >= 0 else ''}{mamba_vs_xgb_vel:.4f} vs XGBoost+velocity)")
print(f"  Griffin (raw seq):   AUPRC = {griffin_auprc:.4f}")

print(f"\n--- Interpretation ---")
if mamba_auprc >= xgb_vel_auprc:
    print(f"  ✅ Mamba MATCHES or EXCEEDS XGBoost+velocity WITHOUT any Feature Store!")
    print(f"  ✅ The SSM's internal state replaces {len(velocity_cols)} hand-crafted velocity features.")
    print(f"  ✅ This eliminates the need for Redis/Kafka/Flink infrastructure.")
elif mamba_auprc >= xgb_basic_auprc:
    print(f"  ⚠️  Mamba exceeds XGBoost(basic) but not XGBoost(+velocity).")
    print(f"  ⚠️  SSM captures SOME temporal patterns but not all that velocity features provide.")
    print(f"  ✅ However, Mamba still adds value as a complementary model.")
else:
    print(f"  ❌ Mamba underperforms even XGBoost(basic).")
    print(f"  This suggests the dataset may not have strong sequential patterns.")

print(f"\n--- Cost Comparison ---")
print(f"  Feature Store (velocity):  Redis cluster + Kafka + Flink = ~$50K-200K/year")
print(f"  SSM State Bank:            1 KB per customer × N customers = negligible")
print(f"  Potential savings:         $50K-200K/year in infrastructure costs")

## Step 14 — Streaming State Bank Demo (Claim #3: Latency)

> **Claim**: SSM streaming inference < 10ms per transaction.

Demonstrate real-time fraud detection: process one transaction at a time,
maintaining O(1) state per customer. No Feature Store lookups needed.

In [ ]:
# === State Bank Demo ===
print("=" * 70)
print("  STATE BANK — Real-Time Streaming Demo")
print("=" * 70)

# Pick a customer with fraud in test set
fraud_test_users = df_test[df_test['Is_Fraud'] == 1]['User'].unique()
if len(fraud_test_users) > 0:
    demo_user = fraud_test_users[0]
else:
    demo_user = df_test['User'].unique()[0]

demo_txs = df_test[df_test['User'] == demo_user].sort_values('datetime')

print(f"\nCustomer: User {demo_user}")
print(f"Total test transactions: {len(demo_txs)}")
print(f"Fraud transactions: {demo_txs['Is_Fraud'].sum()}")

# Stream through Mamba
mamba.eval()
state = None
print(f"\n{'Tx#':>4} {'Date':>20} {'Amount':>10} {'MCC':>6} {'Chip':>10} {'Fraud':>6} {'Mamba Risk':>12}")
print("-" * 80)

with torch.no_grad():
    for i, (_, row) in enumerate(demo_txs.iterrows()):
        features = torch.tensor(row[seq_feature_cols].values.astype(np.float32), device=device).unsqueeze(0)
        logit, state = mamba.forward_single_tx(features, state)
        risk = torch.sigmoid(logit).item()

        is_fraud = '⚠ YES' if row['Is_Fraud'] == 1 else '  no'
        risk_bar = '█' * int(risk * 20)
        alert = ' ← ALERT!' if risk > 0.5 else ''

        if i < 10 or row['Is_Fraud'] == 1 or risk > 0.3 or i >= len(demo_txs) - 5:
            print(f"  {i+1:3d} {str(row['datetime']):>20} ${row['Amount_clean']:>9.2f} {row['MCC']:>6} {row['Use Chip']:>10} {is_fraud:>6} {risk:>8.4f} {risk_bar}{alert}")
        elif i == 10:
            print(f"  ... ({len(demo_txs) - 15} more transactions) ...")

# Calculate state memory
state_bytes = sum(
    s['h'].numel() * 4 + s['conv_buf'].numel() * 4
    for s in state
)
print(f"\nState memory per customer: {state_bytes / 1024:.1f} KB")
print(f"For 10M customers: {state_bytes * 10_000_000 / (1024**3):.1f} GB")

## Step 15 — Latency Benchmark (Claim #3 continued)

> **Claim**: SSM inference < 10ms per transaction.

We measure single-transaction inference latency for streaming mode.

In [ ]:
# === Latency Benchmark ===
import time

print("=" * 70)
print("  LATENCY BENCHMARK — Single Transaction Inference")
print("=" * 70)

dummy_input = torch.randn(1, INPUT_DIM, device=device)
n_warmup = 50
n_measure = 200

latency_data = {}

for name, model_obj in [('Mamba', mamba), ('Griffin', griffin), ('GRU', gru)]:
    model_obj.eval()
    state = None

    # Warmup
    with torch.no_grad():
        for _ in range(n_warmup):
            _, state = model_obj.forward_single_tx(dummy_input, state)

    # Measure
    times = []
    with torch.no_grad():
        for _ in range(n_measure):
            start = time.perf_counter()
            _, state = model_obj.forward_single_tx(dummy_input, state)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            times.append((time.perf_counter() - start) * 1000)

    mean_ms = np.mean(times)
    p50 = np.percentile(times, 50)
    p99 = np.percentile(times, 99)
    latency_data[name] = {'mean_ms': mean_ms, 'p50_ms': p50, 'p99_ms': p99}
    print(f"  {name}: mean={mean_ms:.2f}ms, p50={p50:.2f}ms, p99={p99:.2f}ms")

# XGBoost latency (single prediction)
xgb_times = []
dummy_xgb = np.random.randn(1, len(xgb_feature_cols)).astype(np.float32)
for _ in range(n_measure):
    start = time.perf_counter()
    xgb_vel.predict_proba(dummy_xgb)
    xgb_times.append((time.perf_counter() - start) * 1000)

latency_data['XGBoost'] = {
    'mean_ms': np.mean(xgb_times),
    'p50_ms': np.percentile(xgb_times, 50),
    'p99_ms': np.percentile(xgb_times, 99)
}
print(f"  XGBoost: mean={np.mean(xgb_times):.2f}ms, p50={np.percentile(xgb_times, 50):.2f}ms, p99={np.percentile(xgb_times, 99):.2f}ms")
print(f"  Note: XGBoost latency does NOT include Feature Store lookup time (~50-100ms)")

print(f"\n--- Summary ---")
print(f"  SSM total latency:     ~{latency_data['Mamba']['p50_ms']:.1f}ms (model only, no Feature Store needed)")
print(f"  XGBoost total latency: ~{latency_data['XGBoost']['p50_ms']:.1f}ms model + 50-100ms Feature Store = ~{latency_data['XGBoost']['p50_ms'] + 75:.0f}ms")
print(f"  Production target:     <50ms per transaction")

## Step 16 — False Positive Reduction Analysis (Claim #5)

> **Claim**: Ensemble reduces false positives vs either model alone.

Banks lose customers due to false declines. We measure how the ensemble
improves precision (fewer false alarms) at the same recall level.

In [ ]:
# === False Positive Analysis ===
print("=" * 70)
print("  FALSE POSITIVE REDUCTION ANALYSIS")
print("=" * 70)

from sklearn.metrics import precision_recall_curve

# Compare at fixed recall levels
for target_recall in [0.50, 0.70, 0.90]:
    print(f"\n  @ {target_recall*100:.0f}% Recall:")
    print(f"  {'Model':<25} {'Precision':>10} {'False Positives':>16}")
    print(f"  {'-'*55}")

    for name, probs in [('Mamba (SSM)', pred_mamba),
                        ('XGBoost (+vel)', xgb_vel_probs),
                        ('Mamba+XGB(vel)', ens_mamba_test)]:
        precision, recall, thresholds = precision_recall_curve(y_test, probs)
        # Find threshold closest to target recall
        valid = np.where(recall >= target_recall)[0]
        if len(valid) > 0:
            best_idx = valid[-1]
            p = precision[best_idx]
            r = recall[best_idx]
            thr = thresholds[best_idx] if best_idx < len(thresholds) else 0
            # Count FP
            preds = (probs >= thr).astype(int)
            fp = ((preds == 1) & (y_test == 0)).sum()
            tp = ((preds == 1) & (y_test == 1)).sum()
            print(f"  {name:<25} {p:>10.4f} {fp:>16,}")
        else:
            print(f"  {name:<25} {'N/A':>10} {'N/A':>16}")

print(f"\n--- Interpretation ---")
print(f"  Lower false positives = fewer blocked legitimate customers")
print(f"  = less analyst workload = better customer experience")

In [ ]:
# === Disagreement Analysis ===
thr_m_5 = threshold_at_fpr(y_test, pred_mamba, 0.05)
thr_x_5 = threshold_at_fpr(y_test, xgb_vel_probs, 0.05)

mamba_flags = pred_mamba >= thr_m_5
xgb_flags = xgb_vel_probs >= thr_x_5

agree_fraud = (mamba_flags & xgb_flags & (y_test == 1)).sum()
agree_legit = (~mamba_flags & ~xgb_flags & (y_test == 0)).sum()
only_mamba_correct = (mamba_flags & ~xgb_flags & (y_test == 1)).sum()
only_xgb_correct = (~mamba_flags & xgb_flags & (y_test == 1)).sum()
only_mamba_wrong = (mamba_flags & ~xgb_flags & (y_test == 0)).sum()
only_xgb_wrong = (~mamba_flags & xgb_flags & (y_test == 0)).sum()

print("Disagreement Analysis @ 5% FPR:")
print(f"  Both flag fraud correctly:     {agree_fraud}")
print(f"  Both agree legit:              {agree_legit}")
print(f"  Only Mamba catches (correct):  {only_mamba_correct} ← SSM unique value")
print(f"  Only XGBoost catches (correct):{only_xgb_correct}")
print(f"  Only Mamba false alarm:        {only_mamba_wrong}")
print(f"  Only XGBoost false alarm:      {only_xgb_wrong}")

## Step 17 — Save Comprehensive Results

In [ ]:
# === Save ALL Results ===
import json as json_lib

results_export = {
    'version': '1.0',
    'dataset': 'IBM Credit Card Transactions (ealtman2019/credit-card-transactions)',
    'why_this_dataset': 'Hard (F1=0.86 best published) + Long sequences (~12K tx/user) = ideal for SSM',
    'changes_from_previous': [
        'Dataset: Sparkov → IBM Credit Card Transactions (24.4M tx, 2000 users)',
        'Difficulty: 0.97+ AUPRC (too easy) → 0.86 F1 best published (hard)',
        'Sequences: ~1300 tx/user → ~12000 tx/user',
        'Fraud rate: 0.58% → 0.122% (more realistic)',
        'Added: Infrastructure savings analysis',
        'Added: False positive reduction analysis',
        'Added: XGBoost latency includes Feature Store overhead',
    ],
    'config': {
        'n_users': N_USERS,
        'seq_len': SEQ_LEN,
        'd_model': D_MODEL,
        'n_layers': N_LAYERS,
        'dropout': DROPOUT,
    },
    'data': {
        'total_transactions': len(df),
        'train_transactions': len(df_train),
        'val_transactions': len(df_val),
        'test_transactions': len(df_test),
        'train_sequences': len(y_train),
        'val_sequences': len(y_val),
        'test_sequences': len(y_test),
        'n_users': N_USERS,
        'seq_features': len(seq_feature_cols),
        'velocity_features': len(velocity_cols),
        'fraud_rate_test': float(y_test.mean()),
        'total_fraud_test': int(y_test.sum()),
    },
    'test_results': {
        name: {
            'auprc': float(r['auprc']),
            'auroc': float(r['auroc']),
            'ci_lo': float(r['ci_lo']),
            'ci_hi': float(r['ci_hi']),
            'p_at_1fpr': float(r['p_at_1fpr']),
            'r_at_1fpr': float(r['r_at_1fpr']),
        }
        for name, r in all_results.items()
    },
    'ensemble_weights': {
        'mamba_xgb_vel': float(best_mamba_w),
        'griffin_xgb_vel': float(best_griffin_w),
    },
    'complementarity': complementarity_data,
    'score_correlations_on_fraud': {
        'mamba_vs_xgboost_vel': corr_mx,
        'griffin_vs_xgboost_vel': corr_gx,
        'mamba_vs_griffin': corr_mg,
    },
    'latency': latency_data,
}

save_dir = '/content' if os.path.exists('/content') else '.'
save_path = os.path.join(save_dir, 'results.json')
with open(save_path, 'w') as f:
    json_lib.dump(results_export, f, indent=2)

print(f"Results saved to {save_path}")

try:
    from google.colab import files
    files.download(save_path)
except ImportError:
    pass

## Step 18 — Final Summary: All Claims Tested

In [ ]:
# === Final Summary ===
print("=" * 90)
print("  SSM FRAUD DETECTION PoC  — FINAL SUMMARY")
print("  Dataset: IBM Credit Card Transactions (24.4M tx, 2000 users)")
print("  Published benchmark: F1=0.86 (TabFormer, ICASSP 2021)")
print("=" * 90)

print(f"\n{'Model':<25} {'AUPRC':>8} {'AUROC':>8} {'P@1%FPR':>9} {'R@1%FPR':>9}")
print("-" * 65)
for name, r in sorted(all_results.items(), key=lambda x: -x[1]['auprc']):
    print(f"  {name:<23} {r['auprc']:>8.4f} {r['auroc']:>8.4f} {r['p_at_1fpr']:>9.4f} {r['r_at_1fpr']:>9.4f}")

print(f"\n{'='*90}")
print(f"  CLAIM VALIDATION SUMMARY")
print(f"{'='*90}")

c = complementarity_data.get('5pct_fpr', {})
print(f"\n  Claim #1 — Complementarity (SSM catches what XGBoost misses):")
print(f"    Only Mamba catches @ 5% FPR: {c.get('only_mamba', 'N/A')}")
print(f"    Union recall:                {c.get('union_recall', 0)*100:.1f}%")
print(f"    Score correlation (fraud):   {corr_mx:.3f}")

print(f"\n  Claim #2 — Infrastructure Savings (SSM replaces Feature Store):")
print(f"    XGBoost (basic):    {all_results['XGBoost (basic)']['auprc']:.4f}")
print(f"    XGBoost (+velocity):{all_results['XGBoost (+velocity)']['auprc']:.4f}")
print(f"    Mamba (raw seq):    {all_results['Mamba (SSM)']['auprc']:.4f}")

print(f"\n  Claim #3 — Latency:")
print(f"    Mamba streaming:    {latency_data['Mamba']['p50_ms']:.1f}ms (no Feature Store)")
print(f"    XGBoost:            {latency_data['XGBoost']['p50_ms']:.1f}ms + ~75ms Feature Store")

print(f"\n  Claim #5 — False Positive Reduction:")
print(f"    Ensemble AUPRC:     {all_results.get('Mamba+XGB(vel)', {}).get('auprc', 'N/A')}")

print(f"\n{'='*90}")
print(f"  Dataset comparison across all PoC versions:")
print(f"{'='*90}")
print(f"  v7.5 (IEEE-CIS):  Hard but short sequences → limited SSM potential")
print(f"  v8.0 (Sparkov):   Long sequences but too easy → meaningless results")
print(f"   (IBM CC):    Hard AND long sequences → THE definitive test")
print(f"{'='*90}")